In [0]:
%run /Workspace/Users/vnvarkhede@gmail.com/databricks-genai-data-analyst-copilot/notebooks/08_integration/01_copilot_orchestrator.py


In [0]:
# ================================================================
# PHASE 23 — END-TO-END PIPELINE VALIDATION
# ================================================================

print("=" * 70)
print("PHASE 23 — END-TO-END PIPELINE VALIDATION")
print("=" * 70)

In [0]:
# ================================================================
# 1. LOAD DEPENDENCIES
# ================================================================

import json
import time
import inspect

print("Dependencies loaded.")

In [0]:
# ================================================================
# 2. PIPELINE CONFIGURATION
# ================================================================

CATALOG = "genai_copilot"

BRONZE_TABLE = "genai_copilot.bronze.sales"
SILVER_TABLE = "genai_copilot.silver.sales"
GOLD_TABLE = "genai_copilot.gold.region_sales"

SQL_QUESTION = (
    "Which region generated the highest revenue?"
)

RAG_QUESTION = (
    "What is the discount policy?"
)

HYBRID_QUESTION = (
    "Which region generated the highest revenue "
    "and what discount policy applies there?"
)

print("Catalog:", CATALOG)
print("Bronze:", BRONZE_TABLE)
print("Silver:", SILVER_TABLE)
print("Gold:", GOLD_TABLE)

In [0]:
# ================================================================
# 4. COPILOT DEPENDENCY CHECK
# ================================================================

print("=" * 70)
print("COPILOT DEPENDENCY CHECK")
print("=" * 70)

required_functions = [
    "classify_question",
    "generate_sql_request",
    "validate_sql",
    "execute_sql",
    "generate_question_embedding",
    "retrieve_documents",
    "build_rag_context",
    "generate_rag_answer",
    "decompose_hybrid_question",
    "run_sql_route",
    "run_hybrid_route",
    "assemble_hybrid_answer",
    "ask_copilot"
]

failed_dependencies = []

for function_name in required_functions:

    available = callable(
        globals().get(function_name)
    )

    print(
        f"{'PASS' if available else 'FAIL'} - "
        f"{function_name}"
    )

    if not available:
        failed_dependencies.append(
            function_name
        )

print()
print(
    "Total dependencies:",
    len(required_functions)
)

print(
    "Failed dependencies:",
    len(failed_dependencies)
)

if failed_dependencies:

    raise RuntimeError(
        "Missing dependencies: "
        + ", ".join(
            failed_dependencies
        )
    )

print()
print("Copilot dependency check: PASS")

In [0]:
# ================================================================
# 5. RESPONSE FORMATTER
# ================================================================

def format_copilot_response(response):

    if response is None:

        return {
            "success": False,
            "question": None,
            "route": None,
            "answer": None,
            "sql": None,
            "data": None,
            "sources": [],
            "error": "Copilot returned None.",
            "execution_time_ms": None
        }

    if not isinstance(response, dict):

        return {
            "success": False,
            "question": None,
            "route": None,
            "answer": None,
            "sql": None,
            "data": None,
            "sources": [],
            "error": (
                "Invalid response type: "
                + type(response).__name__
            ),
            "execution_time_ms": None
        }

    return {
        "success": response.get(
            "success",
            False
        ),
        "question": response.get(
            "question"
        ),
        "route": response.get(
            "route"
        ),
        "answer": response.get(
            "answer"
        ),
        "sql": response.get(
            "sql"
        ),
        "data": response.get(
            "data"
        ),
        "sources": response.get(
            "sources",
            []
        ),
        "error": response.get(
            "error"
        ),
        "execution_time_ms": response.get(
            "execution_time_ms"
        )
    }


print(
    "format_copilot_response(): PASS"
)

Tests


In [0]:
# ================================================================
# 6. END-TO-END TEST — SQL ROUTE
# ================================================================

print("=" * 70)
print("END-TO-END TEST — SQL ROUTE")
print("=" * 70)

sql_question = (
    "Which region generated the highest revenue?"
)

sql_response = ask_copilot(
    sql_question
)

sql_result = format_copilot_response(
    sql_response
)

print(
    json.dumps(
        {
            "success": sql_result["success"],
            "question": sql_result["question"],
            "route": sql_result["route"],
            "answer": sql_result["answer"],
            "sql": sql_result["sql"],
            "error": sql_result["error"],
            "execution_time_ms":
                sql_result["execution_time_ms"]
        },
        indent=2,
        default=str
    )
)

if not sql_result["success"]:

    raise RuntimeError(
        "SQL end-to-end test failed: "
        + str(sql_result["error"])
    )

if sql_result["route"] != "sql":

    raise RuntimeError(
        "Expected SQL route but received: "
        + str(sql_result["route"])
    )

print()
print("SQL END-TO-END TEST: PASS")

In [0]:
# ================================================================
# 7. END-TO-END TEST — RAG ROUTE
# ================================================================

print("=" * 70)
print("END-TO-END TEST — RAG ROUTE")
print("=" * 70)

rag_question = (
    "What is the discount policy?"
)

rag_response = ask_copilot(
    rag_question
)

rag_result = format_copilot_response(
    rag_response
)

print(
    json.dumps(
        {
            "success": rag_result["success"],
            "question": rag_result["question"],
            "route": rag_result["route"],
            "answer": rag_result["answer"],
            "sources": rag_result["sources"],
            "error": rag_result["error"],
            "execution_time_ms":
                rag_result["execution_time_ms"]
        },
        indent=2,
        default=str
    )
)

if not rag_result["success"]:

    raise RuntimeError(
        "RAG end-to-end test failed: "
        + str(rag_result["error"])
    )

if rag_result["route"] != "rag":

    raise RuntimeError(
        "Expected RAG route but received: "
        + str(rag_result["route"])
    )

print()
print("RAG END-TO-END TEST: PASS")

In [0]:
# ================================================================
# 8. END-TO-END TEST — HYBRID ROUTE
# ================================================================

print("=" * 70)
print("END-TO-END TEST — HYBRID ROUTE")
print("=" * 70)

hybrid_question = (
    "Which region generated the highest revenue "
    "and what discount policy applies there?"
)

hybrid_response = ask_copilot(
    hybrid_question
)

hybrid_result = format_copilot_response(
    hybrid_response
)

print(
    json.dumps(
        {
            "success": hybrid_result["success"],
            "question": hybrid_result["question"],
            "route": hybrid_result["route"],
            "answer": hybrid_result["answer"],
            "sql": hybrid_result["sql"],
            "error": hybrid_result["error"],
            "execution_time_ms":
                hybrid_result["execution_time_ms"]
        },
        indent=2,
        default=str
    )
)

if not hybrid_result["success"]:

    raise RuntimeError(
        "Hybrid end-to-end test failed: "
        + str(hybrid_result["error"])
    )

if hybrid_result["route"] != "hybrid":

    raise RuntimeError(
        "Expected hybrid route but received: "
        + str(hybrid_result["route"])
    )

print()
print("HYBRID END-TO-END TEST: PASS")

Final Validation

In [0]:
# ================================================================
# 9. FINAL END-TO-END VALIDATION
# ================================================================

print("=" * 70)
print("PHASE 23 — END-TO-END VALIDATION")
print("=" * 70)

end_to_end_tests = [
    ("SQL", sql_result),
    ("RAG", rag_result),
    ("HYBRID", hybrid_result)
]

successful_tests = 0

for route_name, result in end_to_end_tests:

    passed = (
        result.get("success") is True
        and result.get("route") == route_name.lower()
        and result.get("error") is None
    )

    print(
        f"{'PASS' if passed else 'FAIL'} - "
        f"{route_name}"
    )

    if passed:
        successful_tests += 1

print()
print(
    "Total end-to-end tests:",
    len(end_to_end_tests)
)

print(
    "Successful tests:",
    successful_tests
)

print(
    "Failed tests:",
    len(end_to_end_tests) - successful_tests
)

print()

if successful_tests == len(end_to_end_tests):

    print(
        "PHASE 23 STATUS: PASS ✓"
    )

else:

    print(
        "PHASE 23 STATUS: FAIL ✗"
    )

    raise RuntimeError(
        "End-to-end validation failed."
    )